In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Use the Python executable from the active environment
python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

# Prefer local loopback for Spark networking
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"


builder = (
    SparkSession.builder
    .appName("DeltaLakeDemo")
    .master("local[*]")
    .config("spark.pyspark.python", python_path)
    .config("spark.pyspark.driver.python", python_path)
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark Version:", spark.version)
print("Spark session created successfully with Delta Lake support.")

In [ ]:
# Step 1: Create sample data
data = [
    (101, "Amit", 50000),
    (102, "Neha", 60000),
    (103, "Rahul", 55000)
]

columns = ["EmployeeID", "Name", "Salary"]

df = spark.createDataFrame(data, columns)

# Step 2: Write data in Delta format
delta_path = "data/delta/employees"

df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)

print("Delta data written successfully")

# Step 3: Read Delta data
delta_df = spark.read \
    .format("delta") \
    .load(delta_path)

delta_df.show()

# Step 4: Stop Spark
# spark.stop()

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType
)

schema = StructType([
    StructField("OrderID", IntegerType(), False),
    StructField("CustomerName", StringType(), True),
    StructField("Product", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Price", DoubleType(), True),
    StructField("City", StringType(), True)
])

In [ ]:
data = [
    (101, "Amit", "Laptop", 1, 55000.00, "Mumbai"),
    (102, "Neha", "Mouse", 2, 750.00, "Pune"),
    (103, "Rahul", "Keyboard", 1, 1500.00, "Delhi"),
    (104, "Priya", "Monitor", 2, 12000.00, "Mumbai"),
    (105, "Ravi", "Laptop", 1, 60000.00, "Bangalore"),
    (106, "Rajveer Pratap Singh Chauhan", "Laptop", 1, 60000.00, "Chikballapore, Bangalore")
]

df = spark.createDataFrame(data, schema)

df.show(truncate=False)
df.printSchema()

In [ ]:
from pyspark.sql.functions import col

sales_df = df.withColumn(
    "TotalAmount",
    col("Quantity") * col("Price")
)

sales_df.show(truncate=False)

In [ ]:
sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("output/delta/sales")

In [ ]:
delta_df = spark.read \
    .format("delta") \
    .load("output/delta/sales")

delta_df.show()

In [ ]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(
    spark,
    "output/delta/sales"
)

In [ ]:
delta_table.update(
    condition="OrderID = 101",
    set={
        "Price": "60000.0",
        "TotalAmount": "Quantity * 60000.0"
    }
)

In [ ]:
spark.read.format("delta") \
    .load("output/delta/sales") \
    .show()

In [ ]:
delta_table.delete(
    condition="OrderID = 102"
)

In [ ]:
spark.read.format("delta") \
    .load("output/delta/sales") \
    .show()

In [ ]:
merge_data = [
    (103, "Rahul", "Mechanical Keyboard", 1, 2500.0, "Delhi", 2500.0),
    (107, "Karan", "Headphones", 2, 2000.0, "Pune", 4000.0)
]

merge_df = spark.createDataFrame(
    merge_data,
    sales_df.schema
)

In [ ]:
delta_table.alias("target") \
    .merge(
        merge_df.alias("source"),
        "target.OrderID = source.OrderID"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

In [ ]:
spark.read.format("delta") \
    .load("output/delta/sales") \
    .orderBy("OrderID") \
    .show()

In [ ]:
delta_table.history().select(
    "version",
    "timestamp",
    "operation"
).show(truncate=False)

In [ ]:
old_df = spark.read \
    .format("delta") \
    .option("versionAsOf", 0) \
    .load("output/delta/sales")

old_df.show()

In [ ]:
from pyspark.sql.functions import lit

updated_df = sales_df.withColumn(
    "PaymentMode",
    lit("Credit Card")
)

updated_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save("output/delta/sales")

In [ ]:
updated_df.show(truncate=False)